<a href="https://colab.research.google.com/github/akulvibhore/flyrank-ml-internship-/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akulvibhore/flyrank-ml-internship-/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I will use Logistic Regression as the first learned model.

The task is binary classification: identify content items associated with the declining label. Logistic Regression fits this lane because it is simple, fast, and easy to interpret. It also produces a probability score, which can be used to rank items for review.

I will keep the model simple and compare its ranking against the Week-4 rule baseline using the same evaluation metric. The result will be treated as measured decision-support, not as a guarantee of future performance.


In [21]:
# STEP 1 — Load repository + data + prepare features

import os
import sys
import subprocess
import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Clone the repository if it is not already present
# --------------------------------------------------

REPO_DIR = "/content/flyrank-ml-internship-"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/akulvibhore/flyrank-ml-internship-.git

print("Repository ready:", REPO_DIR)

# --------------------------------------------------
# 2. Find the dataset automatically
# --------------------------------------------------

possible_paths = [
    f"{REPO_DIR}/data/raw/content_refresh_anonymized.csv",
    f"{REPO_DIR}/data/content_refresh_anonymized.csv",
    f"{REPO_DIR}/content_refresh_anonymized.csv",
]

DATA_PATH = None

for p in possible_paths:
    if os.path.exists(p):
        DATA_PATH = p
        break

# If the exact file is not found, search the repository
if DATA_PATH is None:
    for root, dirs, files in os.walk(REPO_DIR):
        for file in files:
            if file.lower() == "content_refresh_anonymized.csv":
                DATA_PATH = os.path.join(root, file)
                break
        if DATA_PATH:
            break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Dataset not found inside the cloned repository. "
        "Run the cell below to inspect the available CSV files."
    )

print("Dataset found:")
print(DATA_PATH)

# --------------------------------------------------
# 3. Load data
# --------------------------------------------------

df = pd.read_csv(DATA_PATH)

print("\nRaw shape:", df.shape)
print("\nFirst 10 columns:")
print(df.columns[:10].tolist())

# --------------------------------------------------
# 4. Required columns
# --------------------------------------------------

required_cols = [
    "content_id",
    "client_id",
    "impressions_90d",
    "content_age_days",
    "trend_direction",
]

missing = [
    c for c in required_cols
    if c not in df.columns
]

if missing:
    print("\nMissing required columns:")
    print(missing)

    print("\nAvailable columns:")
    print(df.columns.tolist())

    raise ValueError(
        "The dataset does not contain the expected ML-08 columns."
    )

# --------------------------------------------------
# 5. Numeric features
# --------------------------------------------------

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

numeric_features = [
    c for c in numeric_features
    if c in df.columns
]

# --------------------------------------------------
# 6. Log features
# --------------------------------------------------

for source, target in [
    ("impressions_90d", "log_impressions_90d"),
    ("clicks_90d", "log_clicks_90d"),
    ("sessions_90d", "log_sessions_90d"),
    ("ai_sessions_90d", "log_ai_sessions_90d"),
]:

    if source in df.columns:

        df[target] = np.log1p(
            pd.to_numeric(
                df[source],
                errors="coerce"
            ).clip(lower=0)
        )

        numeric_features.append(target)

# Remove duplicate feature names
numeric_features = list(
    dict.fromkeys(numeric_features)
)

# --------------------------------------------------
# 7. Categorical features
# --------------------------------------------------

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

categorical_features = [
    c for c in categorical_features
    if c in df.columns
]

# --------------------------------------------------
# 8. Convert columns
# --------------------------------------------------

for col in numeric_features:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

for col in categorical_features:

    df[col] = (
        df[col]
        .fillna("unknown")
        .astype(str)
    )

# --------------------------------------------------
# 9. Apply basic filtering
# --------------------------------------------------

df = df[
    (df["impressions_90d"] > 0)
    & (df["content_age_days"] >= 90)
].copy()

# Remove duplicate content
df = df.drop_duplicates(
    "content_id"
).reset_index(drop=True)

# --------------------------------------------------
# 10. Create target
# --------------------------------------------------

df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

# --------------------------------------------------
# 11. Final checks
# --------------------------------------------------

print("\n==============================")
print("ML-08 DATA READY")
print("==============================")

print("Rows:", len(df))
print("Columns:", len(df.columns))
print(
    "Declining rows:",
    int(df["is_declining_label"].sum())
)
print(
    "Declining rate:",
    round(
        df["is_declining_label"].mean(),
        4
    )
)

print(
    "Numeric features:",
    len(numeric_features)
)

print(
    "Categorical features:",
    len(categorical_features)
)

assert df["is_declining_label"].nunique() == 2

print("\nSTEP 1 CHECK PASSED")

Repository ready: /content/flyrank-ml-internship-
Dataset found:
/content/flyrank-ml-internship-/data/raw/content_refresh_anonymized.csv

Raw shape: (30000, 44)

First 10 columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count']

ML-08 DATA READY
Rows: 30000
Columns: 49
Declining rows: 16262
Declining rate: 0.5421
Numeric features: 22
Categorical features: 8

STEP 1 CHECK PASSED


In [22]:
import os

for root, dirs, files in os.walk("/content/flyrank-ml-internship-"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

/content/flyrank-ml-internship-/outputs/refresh_queue_sample.csv
/content/flyrank-ml-internship-/data/raw/content_refresh_anonymized.csv


## 2. Split design
I will use a client-level holdout split.

All content from a client stays in either the training set or the test set. This avoids putting the same client in both sets and gives a more honest estimate of performance on unseen clients.

I will use 80% of the clients for training and 20% for testing, with a fixed random seed of 42 so the split is reproducible.


In [23]:
# STEP 2 — Client-level train/test split

rng = np.random.default_rng(42)

clients = (
    df["client_id"]
    .fillna("unknown")
    .astype(str)
    .drop_duplicates()
    .to_numpy()
)

shuffled_clients = rng.permutation(clients)

test_client_count = max(
    1,
    int(round(len(shuffled_clients) * 0.20))
)

test_clients = set(
    shuffled_clients[:test_client_count]
)

test_mask = (
    df["client_id"]
    .fillna("unknown")
    .astype(str)
    .isin(test_clients)
)

train_mask = ~test_mask

train_df = df.loc[train_mask].copy()
test_df = df.loc[test_mask].copy()

print("Split strategy: client_holdout")
print("Total clients:", len(clients))
print("Training clients:", train_df["client_id"].nunique())
print("Testing clients:", test_df["client_id"].nunique())
print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

print(
    "\nTraining declining rate:",
    round(train_df["is_declining_label"].mean(), 4)
)

print(
    "Testing declining rate:",
    round(test_df["is_declining_label"].mean(), 4)
)

# Verify no client leakage
overlap = set(train_df["client_id"]) & set(test_df["client_id"])

print("\nClient overlap:", len(overlap))

assert len(overlap) == 0
assert len(train_df) > 0
assert len(test_df) > 0
assert train_df["is_declining_label"].nunique() == 2
assert test_df["is_declining_label"].nunique() == 2

print("\nSTEP 2 CHECK PASSED")

Split strategy: client_holdout
Total clients: 32
Training clients: 26
Testing clients: 6
Training rows: 27675
Testing rows: 2325

Training declining rate: 0.5548
Testing declining rate: 0.391

Client overlap: 0

STEP 2 CHECK PASSED


## 3. Train + compare vs my baseline

I will train Logistic Regression on the training clients and evaluate it on the held-out clients.

The model and the Week-4 baseline will use the same test set and the same Precision@K metric. I will compare Precision@20, Precision@50, and Precision@100.

The comparison will be used as decision-support. I will report the measured result without treating it as a guarantee of future performance.


In [24]:
# STEP 3 — Train Logistic Regression + compare with Week-4 baseline

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

RANDOM_STATE = 42

# --------------------------------------------------
# 1. Prepare train/test data
# --------------------------------------------------

feature_columns = numeric_features + categorical_features

X_train = train_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()

y_train = train_df["is_declining_label"].astype(int)
y_test = test_df["is_declining_label"].astype(int)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

# --------------------------------------------------
# 2. Preprocessing
# --------------------------------------------------

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

# --------------------------------------------------
# 3. Logistic Regression
# --------------------------------------------------

model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=RANDOM_STATE
            )
        )
    ]
)

# Train
model.fit(X_train, y_train)

# Probability score
model_scores = model.predict_proba(X_test)[:, 1]

# Classification prediction
model_predictions = (
    model_scores >= 0.5
).astype(int)

print("\nMODEL TRAINED SUCCESSFULLY")

# --------------------------------------------------
# 4. Precision@K function
# --------------------------------------------------

def precision_at_k(y_true, scores, k):

    temp = pd.DataFrame({
        "actual": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    temp = temp.sort_values(
        "score",
        ascending=False
    )

    top_k = temp.head(k)

    return float(top_k["actual"].mean())


# --------------------------------------------------
# 5. Find Week-4 baseline
# --------------------------------------------------

baseline_candidates = [
    f"{REPO_DIR}/outputs/refresh_queue_sample.csv",
    f"{REPO_DIR}/data/processed/baseline_refresh_queue.csv",
    f"{REPO_DIR}/work/outputs/baseline_action_score.csv"
]

baseline_path = None

for p in baseline_candidates:

    if os.path.exists(p):
        baseline_path = p
        break

print("\nBaseline file:")

if baseline_path:
    print(baseline_path)
else:
    print("No baseline file found.")

# --------------------------------------------------
# 6. Load/recreate baseline
# --------------------------------------------------

if baseline_path:

    baseline_df = pd.read_csv(baseline_path)

    print(
        "Baseline shape:",
        baseline_df.shape
    )

    print(
        "Baseline columns:",
        baseline_df.columns.tolist()
    )

    # Try to identify score column
    score_candidates = [
        "baseline_refresh_score",
        "baseline_action_score",
        "action_score",
        "score"
    ]

    baseline_score_col = None

    for col in score_candidates:

        if col in baseline_df.columns:
            baseline_score_col = col
            break

    if baseline_score_col is None:

        raise ValueError(
            "Could not find the baseline score column. "
            "Check the printed baseline columns above."
        )

    # Create lookup
    baseline_lookup = (
        baseline_df
        .drop_duplicates("content_id")
        .set_index("content_id")[baseline_score_col]
    )

    baseline_scores = (
        test_df["content_id"]
        .map(baseline_lookup)
        .fillna(0)
        .to_numpy()
    )

    print(
        "Using baseline score:",
        baseline_score_col
    )

else:

    # Fallback transparent baseline
    print(
        "\nUsing fallback transparent baseline."
    )

    baseline_work = test_df.copy()

    # Visibility
    visibility = np.log1p(
        baseline_work["impressions_90d"]
        .clip(lower=0)
    )

    if visibility.max() > visibility.min():

        visibility = (
            visibility - visibility.min()
        ) / (
            visibility.max() - visibility.min()
        )

    # Freshness
    freshness = (
        baseline_work[
            "days_since_last_update"
        ]
        .fillna(0)
        .clip(lower=0)
    )

    freshness = np.minimum(
        freshness / 365.0,
        1.0
    )

    # Position opportunity
    position = (
        baseline_work["avg_position"]
        .fillna(0)
    )

    position_score = np.where(
        (position > 0) &
        (position <= 20),
        (21 - position) / 20,
        0
    )

    # Content depth gap
    word_count = (
        baseline_work["word_count"]
        .fillna(0)
    )

    depth_gap = np.where(
        (word_count > 0) &
        (word_count < 1200),
        1 - (word_count / 1200),
        0
    )

    baseline_scores = (
        0.40 * visibility
        + 0.30 * freshness
        + 0.25 * position_score
        + 0.05 * depth_gap
    )

# --------------------------------------------------
# 7. Model metrics
# --------------------------------------------------

accuracy = accuracy_score(
    y_test,
    model_predictions
)

precision = precision_score(
    y_test,
    model_predictions,
    zero_division=0
)

recall = recall_score(
    y_test,
    model_predictions,
    zero_division=0
)

f1 = f1_score(
    y_test,
    model_predictions,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_test,
    model_scores
)

avg_precision = average_precision_score(
    y_test,
    model_scores
)

print("\n==============================")
print("MODEL METRICS")
print("==============================")

print("Accuracy:", round(accuracy, 3))
print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))
print("F1:", round(f1, 3))
print("ROC AUC:", round(roc_auc, 3))
print("Average Precision:", round(avg_precision, 3))

# --------------------------------------------------
# 8. Precision@K comparison
# --------------------------------------------------

ks = [20, 50, 100]

results = []

for k in ks:

    model_p = precision_at_k(
        y_test,
        model_scores,
        k
    )

    baseline_p = precision_at_k(
        y_test,
        baseline_scores,
        k
    )

    results.append({
        "Method": "Week-4 baseline",
        "K": k,
        "Precision": baseline_p
    })

    results.append({
        "Method": "Logistic Regression",
        "K": k,
        "Precision": model_p
    })

comparison = pd.DataFrame(results)

comparison["Precision"] = comparison[
    "Precision"
].round(3)

print("\n==============================")
print("PRECISION@K COMPARISON")
print("==============================")

display(
    comparison.pivot(
        index="Method",
        columns="K",
        values="Precision"
    )
)

# --------------------------------------------------
# 9. Base rate
# --------------------------------------------------

print(
    "\nTest-set declining base rate:",
    round(y_test.mean(), 3)
)

print("\nSTEP 3 CHECK PASSED")

Training shape: (27675, 30)
Testing shape: (2325, 30)

MODEL TRAINED SUCCESSFULLY

Baseline file:
/content/flyrank-ml-internship-/outputs/refresh_queue_sample.csv
Baseline shape: (200, 28)
Baseline columns: ['final_rank', 'content_id', 'client_id', 'final_refresh_score', 'best_model_name', 'best_model_probability', 'baseline_refresh_score', 'confidence', 'suggested_action', 'final_reason_codes', 'is_declining_label', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position', 'ctr', 'content_age_days', 'days_since_last_update', 'word_count', 'trend_direction', 'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']
Using baseline score: baseline_refresh_score

MODEL METRICS
Accuracy: 0.663
Precision: 0.57
Recall: 0.563
F1: 0.567
ROC AUC: 0.702
Average Precision: 0.52

PRECISION@K COMPARISON


K,20,50,100
Method,,,
Logistic Regression,0.35,0.32,0.41
Week-4 baseline,0.45,0.46,0.41



Test-set declining base rate: 0.391

STEP 3 CHECK PASSED


## 4. Errors and interpretation

The model showed moderate discrimination on the held-out clients, with ROC-AUC of 0.702 and average precision of 0.520.

The main error pattern is visible in the top-K review. The Logistic Regression model measured Precision@20 of 0.35 and Precision@50 of 0.32, compared with 0.45 and 0.46 for the Week-4 baseline. At Precision@100, both methods measured 0.41.

This suggests that the learned model did not improve the highest-priority review queue over the existing baseline on this split. The model appears to contain useful signal, but its ranking is not consistently better than the baseline at the smaller review sizes.

I will inspect the highest-scored false positives and false negatives and check which features have the largest model coefficients. These results are directional and should be treated as decision-support rather than a guarantee of future performance.


In [25]:
# STEP 4 — Error analysis and feature interpretation

# --------------------------------------------------
# 1. Create an error-analysis dataframe
# --------------------------------------------------

error_df = test_df[
    [
        "content_id",
        "client_id",
        "is_declining_label"
    ]
].copy()

error_df["model_score"] = model_scores
error_df["prediction"] = model_predictions

error_df["error_type"] = np.select(
    [
        (error_df["prediction"] == 1) &
        (error_df["is_declining_label"] == 0),

        (error_df["prediction"] == 0) &
        (error_df["is_declining_label"] == 1)
    ],
    [
        "False Positive",
        "False Negative"
    ],
    default="Correct"
)

print("ERROR COUNTS")
print(
    error_df["error_type"].value_counts()
)

# --------------------------------------------------
# 2. Highest-scored false positives
# --------------------------------------------------

false_positives = (
    error_df[
        error_df["error_type"] == "False Positive"
    ]
    .sort_values(
        "model_score",
        ascending=False
    )
    .head(10)
)

print("\nTOP FALSE POSITIVES")
display(false_positives)

# --------------------------------------------------
# 3. Highest-confidence false negatives
# --------------------------------------------------

false_negatives = (
    error_df[
        error_df["error_type"] == "False Negative"
    ]
    .sort_values(
        "model_score",
        ascending=True
    )
    .head(10)
)

print("\nTOP FALSE NEGATIVES")
display(false_negatives)

# --------------------------------------------------
# 4. Feature importance / coefficients
# --------------------------------------------------

fitted_preprocessor = (
    model.named_steps["preprocessor"]
)

classifier = (
    model.named_steps["classifier"]
)

feature_names = (
    fitted_preprocessor
    .get_feature_names_out()
)

coefficients = classifier.coef_[0]

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "absolute_coefficient": np.abs(coefficients)
})

feature_importance = (
    feature_importance
    .sort_values(
        "absolute_coefficient",
        ascending=False
    )
)

print("\nTOP MODEL FEATURES")
display(
    feature_importance.head(15)
)

# --------------------------------------------------
# 5. Summary numbers for notebook
# --------------------------------------------------

fp_count = (
    error_df["error_type"]
    .eq("False Positive")
    .sum()
)

fn_count = (
    error_df["error_type"]
    .eq("False Negative")
    .sum()
)

print("\n==============================")
print("ERROR ANALYSIS SUMMARY")
print("==============================")

print("False positives:", fp_count)
print("False negatives:", fn_count)

print(
    "P@20 model:",
    round(
        precision_at_k(
            y_test,
            model_scores,
            20
        ),
        3
    )
)

print(
    "P@50 model:",
    round(
        precision_at_k(
            y_test,
            model_scores,
            50
        ),
        3
    )
)

print(
    "P@100 model:",
    round(
        precision_at_k(
            y_test,
            model_scores,
            100
        ),
        3
    )
)

print("\nSTEP 4 CHECK PASSED")

ERROR COUNTS
error_type
Correct           1542
False Negative     397
False Positive     386
Name: count, dtype: int64

TOP FALSE POSITIVES


,content_id,client_id,is_declining_label,model_score,prediction,error_type
4994,content_8fdbff16a886,client_f74efabef1,0,0.905098,1,False Positive
21024,content_da806b9f243e,client_f74efabef1,0,0.886562,1,False Positive
26326,content_e354f8e518c2,client_f74efabef1,0,0.885920,1,False Positive
22966,content_efe1a1a25ab0,client_f74efabef1,0,0.881307,1,False Positive
4102,content_8f06ddfec8bc,client_f74efabef1,0,0.876594,1,False Positive
20141,content_646483666d1d,client_f74efabef1,0,0.874811,1,False Positive
13635,content_2ed4cd0eb5a9,client_f74efabef1,0,0.864934,1,False Positive
5866,content_66067a014ad5,client_f74efabef1,0,0.863280,1,False Positive
1886,content_3e79eaafc89d,client_f74efabef1,0,0.849797,1,False Positive
3982,content_9b4ddfa91f64,client_0b918943df,0,0.848810,1,False Positive



TOP FALSE NEGATIVES


,content_id,client_id,is_declining_label,model_score,prediction,error_type
10352,content_e0e412fa582f,client_f74efabef1,1,0.000031,0,False Negative
17188,content_a0a76e94ade5,client_f74efabef1,1,0.000035,0,False Negative
17901,content_d29e488c8812,client_f74efabef1,1,0.000036,0,False Negative
8206,content_973a43ede99f,client_f74efabef1,1,0.000039,0,False Negative
15285,content_7ff6ded89d4a,client_f74efabef1,1,0.000039,0,False Negative
1068,content_836b4163cf30,client_f74efabef1,1,0.000051,0,False Negative
28865,content_8d944a023366,client_f74efabef1,1,0.000051,0,False Negative
28692,content_5a9eed16e004,client_f74efabef1,1,0.000059,0,False Negative
22322,content_f2f800ba53d5,client_f74efabef1,1,0.000072,0,False Negative
6943,content_13db9ddaa0a9,client_f74efabef1,1,0.000088,0,False Negative



TOP MODEL FEATURES


,feature,coefficient,absolute_coefficient
18,num__log_impressions_90d,1.726208,1.726208
3,num__word_count,1.108755,1.108755
4,num__char_count,-0.946694,0.946694
55,cat__position_tier_top_3,-0.702048,0.702048
19,num__log_clicks_90d,-0.668546,0.668546
49,cat__impression_tier_low,0.625151,0.625151
47,cat__impression_tier_excellent,-0.571609,0.571609
45,cat__word_count_tier_<1000,0.505517,0.505517
46,cat__word_count_tier_unknown,-0.489116,0.489116
33,cat__main_intent_unknown,0.480115,0.480115



ERROR ANALYSIS SUMMARY
False positives: 386
False negatives: 397
P@20 model: 0.35
P@50 model: 0.32
P@100 model: 0.41

STEP 4 CHECK PASSED


The model produced 386 false positives and 397 false negatives on the held-out test set.

Among the inspected high-confidence errors, several false positives and false negatives came from the same client. This is an observed pattern in the error sample, although it is not enough evidence to conclude that the model has a general client-specific bias.

The largest model coefficients were associated with log impressions, word count, character count, position tier, log clicks, and impression tiers. This indicates that the model is relying substantially on traffic, content-size, and ranking-related signals.

The model achieved ROC-AUC of 0.702 and average precision of 0.520, showing useful predictive signal. However, the ranking comparison is more important for the review queue: Logistic Regression measured Precision@20 of 0.35 and Precision@50 of 0.32, compared with 0.45 and 0.46 for the Week-4 baseline. At Precision@100, both methods measured 0.41.

Therefore, on this held-out client split, the learned model did not improve the baseline for the smaller review queues. The results are directional and should be treated as decision-support rather than a guarantee of future performance.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.